# Exploratory Data Analysis — Template

First-pass EDA template for any classical tabular dataset.

**How to use**
- Set `DATA_PATH` in the *Configuration* cell and run all cells top to bottom.
- All helper functions live in this notebook (no external `.py` file).
- Plots are styled with the **Wavestone template color palette** so screenshots can be pasted directly into the Google Slides deck.

**Sections**
1. Configuration & palette
2. Loading & overview
3. Univariate analysis (numeric & categorical)
4. Bivariate analysis (correlations, numeric vs numeric, numeric vs categorical)
5. Quick takeaways

## 1. Configuration & Wavestone palette

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Dataset path — adapt to your project (csv, parquet, excel...)
# ---------------------------------------------------------------------------
DATA_PATH = "../data/dataset.csv"

# Set to True to run the template on a synthetic demo dataset instead
USE_DEMO_DATA = True

# ---------------------------------------------------------------------------
# Wavestone template palette (match the Google Slides template)
# ---------------------------------------------------------------------------
COLORS = {
    "primary": "#3C18AE",      # Wavestone deep purple
    "secondary": "#441DC6",    # Wavestone bright purple
    "accent": "#8B7FF0",       # light purple
    "neutral": "#DFD6C5",      # sand / beige
    "neutral_dark": "#C0B5B3", # warm grey
    "bg_light": "#EBEEFF",     # very light blue-violet
    "dark": "#1A1A2E",         # anthracite (titles / contrast)
    "grey": "#6B6B7B",         # secondary text
    "positive": "#2E8B57",     # green (positive signal)
    "negative": "#B23A48",     # red (negative signal)
}

SEQUENCE = [
    COLORS["primary"], COLORS["accent"], COLORS["neutral"],
    COLORS["secondary"], COLORS["neutral_dark"], COLORS["grey"],
]

TEMPLATE_LAYOUT = {
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": {"family": "Calibri, Arial, sans-serif", "size": 13, "color": COLORS["dark"]},
    "title": {"font": {"size": 20, "color": COLORS["dark"]}},
    "colorway": SEQUENCE,
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = SEQUENCE
px.defaults.color_continuous_scale = [COLORS["neutral"], COLORS["accent"], COLORS["primary"]]


def style_fig(fig: go.Figure, title: str, width: int = 900, height: int = 480) -> go.Figure:
    """Apply Wavestone template styling to a Plotly figure."""
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        template={"layout": TEMPLATE_LAYOUT},
        width=width,
        height=height,
        margin={"l": 60, "r": 40, "t": 70, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 1, "xanchor": "right"},
    )
    fig.update_xaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    return fig

## 2. Load data & first overview

In [ ]:
def load_data(path: str = DATA_PATH, demo: bool = USE_DEMO_DATA) -> pd.DataFrame:
    """Load the dataset from path, or generate a synthetic demo dataset."""
    if demo:
        rng = np.random.default_rng(42)
        n = 800
        segment = rng.choice(["SME", "Mid-cap", "Large-cap"], size=n, p=[0.5, 0.3, 0.2])
        base = {"SME": 40, "Mid-cap": 60, "Large-cap": 90}["SME"]
        revenue = (
            rng.normal(50, 15, size=n)
            + pd.Series(segment).map({"SME": 0, "Mid-cap": 25, "Large-cap": 60}).values
        )
        df = pd.DataFrame({
            "customer_id": [f"C{str(i).zfill(4)}" for i in range(n)],
            "segment": segment,
            "region": rng.choice(["France", "Germany", "UK", "Switzerland"], size=n),
            "annual_revenue_k": np.round(np.clip(revenue, 5, None), 1),
            "contract_years": rng.integers(1, 10, size=n),
            "satisfaction_score": np.round(rng.uniform(1, 10, size=n), 1),
            "churned": rng.choice([0, 1], size=n, p=[0.7, 0.3]),
        })
        df.loc[df.sample(frac=0.02, random_state=1).index, "annual_revenue_k"] = np.nan
        return df
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".xlsx"):
        return pd.read_excel(path)
    return pd.read_csv(path)


df = load_data()
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
def overview(df: pd.DataFrame) -> pd.DataFrame:
    """Build a per-column overview table: dtype, missing, unique, sample values."""
    rows = []
    for col in df.columns:
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "n_missing": int(df[col].isna().sum()),
            "pct_missing": round(100 * df[col].isna().mean(), 2),
            "n_unique": int(df[col].nunique()),
            "sample_values": ", ".join(map(str, df[col].dropna().unique()[:3])),
        })
    return pd.DataFrame(rows)


overview_df = overview(df)
overview_df

In [ ]:
def plot_missing(df: pd.DataFrame) -> go.Figure:
    """Bar chart of missing values per column."""
    miss = df.isna().mean().mul(100).sort_values(ascending=False)
    miss = miss[miss > 0]
    if miss.empty:
        print("No missing values in the dataset.")
        return go.Figure()
    fig = px.bar(x=miss.index, y=miss.values, labels={"x": "", "y": "% missing"})
    fig.update_traces(marker_color=COLORS["primary"])
    return style_fig(fig, "Missing values per column (%)", height=380)


if df.isna().any().any():
    plot_missing(df).show()
else:
    print("No missing values.")

In [ ]:
def describe_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Extended descriptive statistics for numeric columns."""
    num = df.select_dtypes(include=[np.number])
    if num.empty:
        return pd.DataFrame()
    desc = num.describe().T
    desc["skew"] = num.skew()
    desc["kurtosis"] = num.kurtosis()
    return desc.round(2)


describe_numeric(df)

## 3. Univariate analysis

In [ ]:
def split_columns(df: pd.DataFrame, max_cat: int = 15) -> tuple[list[str], list[str]]:
    """Split columns into numeric and categorical (low-cardinality) lists."""
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [
        c for c in df.columns
        if c not in numeric and df[c].nunique() <= max_cat
    ]
    return numeric, categorical


NUMERIC_COLS, CATEGORICAL_COLS = split_columns(df)
print("Numeric:", NUMERIC_COLS)
print("Categorical:", CATEGORICAL_COLS)

In [ ]:
def plot_histograms(df: pd.DataFrame, cols: list[str], color: str | None = None) -> None:
    """One styled histogram (+ box overlay toggle off) per numeric column."""
    for col in cols:
        fig = px.histogram(df, x=col, color=color, marginal="box", nbins=30, barmode="overlay")
        fig.update_traces(marker_line_width=0.5, opacity=0.85)
        style_fig(fig, f"Distribution of {col}").show()


plot_histograms(df, NUMERIC_COLS)

In [ ]:
def plot_value_counts(df: pd.DataFrame, cols: list[str]) -> None:
    """Horizontal bar chart of category frequencies per categorical column."""
    for col in cols:
        counts = df[col].value_counts().sort_values(ascending=True)
        fig = px.bar(x=counts.values, y=counts.index, orientation="h",
                     labels={"x": "count", "y": col})
        fig.update_traces(marker_color=COLORS["primary"], showlegend=False)
        style_fig(fig, f"Value counts — {col}", height=360).show()


plot_value_counts(df, CATEGORICAL_COLS)

## 4. Bivariate analysis

In [ ]:
def plot_correlation_matrix(df: pd.DataFrame, cols: list[str] | None = None) -> go.Figure:
    """Pearson correlation heatmap on numeric columns."""
    num = df[cols] if cols else df.select_dtypes(include=[np.number])
    corr = num.corr(method="pearson")
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale=[[0, COLORS["negative"]], [0.5, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=-1, zmax=1, text=corr.round(2).values, texttemplate="%{text}",
    ))
    return style_fig(fig, "Correlation matrix (Pearson)", width=750, height=600)


plot_correlation_matrix(df).show()

In [ ]:
def plot_scatter(df: pd.DataFrame, x: str, y: str, color: str | None = None,
                 trendline: bool = False) -> go.Figure:
    """Styled scatter plot with optional trendline (requires statsmodels)."""
    fig = px.scatter(df, x=x, y=y, color=color, opacity=0.7,
                     trendline="lowess" if trendline else None)
    fig.update_traces(marker={"size": 7, "line": {"width": 0}})
    return style_fig(fig, f"{y} vs {x}")


# Adapt the column names to your dataset
if len(NUMERIC_COLS) >= 2:
    plot_scatter(df, NUMERIC_COLS[0], NUMERIC_COLS[1], color=CATEGORICAL_COLS[0] if CATEGORICAL_COLS else None).show()

In [ ]:
def plot_box_by_category(df: pd.DataFrame, numeric_cols: list[str], cat_cols: list[str]) -> None:
    """Box plot of each numeric column grouped by each categorical column."""
    for num in numeric_cols:
        for cat in cat_cols:
            fig = px.box(df, x=cat, y=num, color=cat, points="outliers")
            fig.update_traces(showlegend=False)
            style_fig(fig, f"{num} by {cat}", height=420).show()


# Limit to 2 numeric x 2 categorical to keep the notebook readable
plot_box_by_category(df, NUMERIC_COLS[:2], CATEGORICAL_COLS[:2])

In [ ]:
def plot_grouped_aggregate(df: pd.DataFrame, cat: str, num: str, agg: str = "mean") -> go.Figure:
    """Bar chart of an aggregate (mean/median/sum) of num grouped by cat."""
    agg_df = df.groupby(cat, observed=True)[num].agg(agg).sort_values(ascending=False)
    fig = px.bar(x=agg_df.index, y=agg_df.values, labels={"x": cat, "y": f"{agg} of {num}"})
    fig.update_traces(marker_color=COLORS["primary"], showlegend=False)
    return style_fig(fig, f"{agg.capitalize()} {num} by {cat}", height=420)


if NUMERIC_COLS and CATEGORICAL_COLS:
    plot_grouped_aggregate(df, CATEGORICAL_COLS[0], NUMERIC_COLS[0]).show()

## 5. Quick takeaways

Use this section to summarize what the first pass reveals:

- **Data quality**: missing values, duplicates, outliers, suspicious dtypes
- **Distributions**: skewness, multi-modality, constant or near-constant columns
- **Relationships**: strongest correlations, notable group differences
- **Next steps**: cleaning decisions, feature engineering ideas, modeling assumptions to validate